# Singleton recommendation inspection — history MLP ensemble

This notebook loads every compatible `history_mlp` checkpoint under `artifacts/implicit/history_mlp_ensemble/seed_*`, averages member probabilities, and inspects several one-book queries. It deliberately checks more than whether the titles look attractive:

- whether recommendations change with the query;
- whether they collapse to globally popular books;
- how many training readers support each query–recommendation pair.

The displayed sigmoid values are ranking scores learned with sampled BCE. They are not calibrated probabilities that a reader will like a book.

In [19]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from bookrec.data import ITEM_COLUMN, USER_COLUMN, load_dataset, split_interactions
from bookrec.implicit.inference import (
    load_history_mlp,
    predict_history_probabilities,
)

ROOT = Path.cwd()
if not (ROOT / "bookrec").exists():
    ROOT = ROOT.parent

ENSEMBLE_DIRECTORY = ROOT / "artifacts" / "implicit" / "history_mlp_ensemble"
SPLIT_SEED = 42
TOP_K = 5
MIN_QUERY_TRAINING_INTERACTIONS = 20
MIN_RECOMMENDATION_INTERACTIONS = 20
INFERENCE_BATCH_SIZE = 16_384
QUERY_TITLES = [
    "The Lord of the Rings",
    "1984",
    "I, Robot",
    "The Hobbit",
    "Harry Potter and the Sorcerer's Stone",
    "ROMEO AND JULIET",
]

## 1. Load the ensemble and training-only support data

The inference loader validates architecture versions, mappings, and hyperparameters before constructing the probability ensemble. Popularity and co-reader diagnostics use only the training split.

In [20]:
loaded = load_history_mlp(ENSEMBLE_DIRECTORY)
if not loaded.is_ensemble:
    raise ValueError(f"Expected an ensemble in {ENSEMBLE_DIRECTORY}")
if loaded.training_item_counts is None:
    raise ValueError("Checkpoints do not contain training item counts")

ratings = load_dataset()
books = load_dataset("Books.csv")
books[ITEM_COLUMN] = books[ITEM_COLUMN].astype(str)
metadata = books.drop_duplicates(ITEM_COLUMN).copy()

train_raw, validation_raw, test_raw = split_interactions(
    ratings, seed=SPLIT_SEED
)
train_raw = train_raw.copy()
train_raw[ITEM_COLUMN] = train_raw[ITEM_COLUMN].astype(str)

training_counts = pd.Series(
    loaded.training_item_counts.numpy(),
    index=pd.Index(loaded.index_to_item, name=ITEM_COLUMN),
    name="training interactions",
).astype(int)
popularity_rank = training_counts.rank(method="min", ascending=False)
mapped_isbns = set(loaded.item_to_index)
mapped_metadata = metadata[metadata[ITEM_COLUMN].isin(mapped_isbns)].copy()
metadata_isbns = set(mapped_metadata[ITEM_COLUMN])

print(f"Device: {loaded.device}")
print(f"Ensemble members: {len(loaded.member_paths)}")
print(f"Mapped catalog items: {len(loaded.item_to_index):,}")
display(pd.DataFrame({"checkpoint": [str(path) for path in loaded.member_paths]}))

/home/nuva/Job/Datasentics/.venv/lib/python3.13/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


Device: cuda
Ensemble members: 5
Mapped catalog items: 303,422


,checkpoint
0,/home/nuva/Job/Datasentics/artifacts/implicit/...
1,/home/nuva/Job/Datasentics/artifacts/implicit/...
2,/home/nuva/Job/Datasentics/artifacts/implicit/...
3,/home/nuva/Job/Datasentics/artifacts/implicit/...
4,/home/nuva/Job/Datasentics/artifacts/implicit/...


## 2. Resolve titles and run singleton inference

Title resolution first looks for an exact-title edition meeting `MIN_QUERY_TRAINING_INTERACTIONS`. If no exact edition has enough support, it searches title variants starting with the requested title and selects the most-interacted supported edition. This permits edition suffixes without matching unrelated titles that merely mention the query. Candidates without metadata or with fewer than `MIN_RECOMMENDATION_INTERACTIONS` are excluded, and the input ISBN is never returned as its own recommendation.

In [21]:
def resolve_title(title):
    titles = mapped_metadata["Book-Title"].astype(str)

    def supported(matches):
        matches = matches.copy()
        matches["training interactions"] = (
            matches[ITEM_COLUMN].map(training_counts).fillna(0).astype(int)
        )
        return matches[
            matches["training interactions"] >= MIN_QUERY_TRAINING_INTERACTIONS
        ]

    exact_matches = supported(
        mapped_metadata[titles.str.casefold() == title.casefold()]
    )
    if not exact_matches.empty:
        return exact_matches.nlargest(1, "training interactions").iloc[0]

    variant_matches = supported(
        mapped_metadata[
            titles.str.casefold().str.startswith(title.casefold(), na=False)
        ]
    )
    if variant_matches.empty:
        return None
    return variant_matches.nlargest(1, "training interactions").iloc[0]


eligible_candidate_indices = torch.tensor(
    [
        index
        for isbn, index in loaded.item_to_index.items()
        if isbn in metadata_isbns
        and training_counts.get(isbn, 0) >= MIN_RECOMMENDATION_INTERACTIONS
    ],
    dtype=torch.long,
)


def recommend_singleton(query_title, top_k=TOP_K):
    query = resolve_title(query_title)
    if query is None:
        return None, None

    query_isbn = query[ITEM_COLUMN]
    query_index = loaded.item_to_index[query_isbn]
    candidates = eligible_candidate_indices[
        eligible_candidate_indices != query_index
    ]
    probabilities = predict_history_probabilities(
        loaded.model,
        history_items=[query_index],
        candidate_items=candidates,
        batch_size=INFERENCE_BATCH_SIZE,
    )

    result_count = min(top_k, len(candidates))
    top_positions = torch.topk(probabilities, result_count).indices
    top_indices = candidates[top_positions].tolist()
    recommendations = pd.DataFrame({
        ITEM_COLUMN: [loaded.index_to_item[index] for index in top_indices],
        "ensemble probability": probabilities[top_positions].numpy(),
        "rank": np.arange(1, result_count + 1),
    }).merge(metadata, on=ITEM_COLUMN, how="left")
    recommendations["training interactions"] = (
        recommendations[ITEM_COLUMN].map(training_counts).astype(int)
    )
    return query, recommendations

In [22]:
query_rows = []
recommendation_frames = []

for requested_title in QUERY_TITLES:
    query, recommendations = recommend_singleton(requested_title)
    if query is None:
        print(
            f"Skipping {requested_title!r}: no mapped title variant has at least "
            f"{MIN_QUERY_TRAINING_INTERACTIONS} training interactions"
        )
        continue

    query_isbn = query[ITEM_COLUMN]
    query_interactions = int(training_counts[query_isbn])
    query_rows.append({
        "requested title": requested_title,
        "resolved title": query["Book-Title"],
        "query ISBN": query_isbn,
        "query training interactions": query_interactions,
    })
    recommendations = recommendations.copy()
    recommendations["query title"] = query["Book-Title"]
    recommendations["query ISBN"] = query_isbn
    recommendation_frames.append(recommendations)

resolved_queries = pd.DataFrame(query_rows)
all_recommendations = pd.concat(recommendation_frames, ignore_index=True)
display(resolved_queries)
display(all_recommendations[[
    "query title",
    "rank",
    "Book-Title",
    "Book-Author",
    ITEM_COLUMN,
    "training interactions",
    "ensemble probability",
]])

,requested title,resolved title,query ISBN,query training interactions
0,The Lord of the Rings,The Lord of the Rings (Movie Art Cover),0618129022,58
1,1984,1984,0451524934,163
2,"I, Robot","I, Robot",0553294385,49
3,The Hobbit,The Hobbit : The Enchanting Prelude to The Lor...,0345339681,231
4,Harry Potter and the Sorcerer's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,059035342X,490
5,ROMEO AND JULIET,ROMEO AND JULIET,0671722859,27


,query title,rank,Book-Title,Book-Author,ISBN,training interactions,ensemble probability
0,The Lord of the Rings (Movie Art Cover),1,The Mists of Avalon,MARION ZIMMER BRADLEY,0345350499,145,0.985950
1,The Lord of the Rings (Movie Art Cover),2,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,043935806X,278,0.978953
2,The Lord of the Rings (Movie Art Cover),3,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0590353403,143,0.972657
3,The Lord of the Rings (Movie Art Cover),4,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,0439064864,141,0.971559
4,The Lord of the Rings (Movie Art Cover),5,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0439139600,157,0.971145
5,1984,1,Brave New World,Aldous Huxley,0060929871,101,0.990794
6,1984,2,Animal Farm,George Orwell,0451526341,132,0.990150
7,1984,3,Slaughterhouse Five or the Children's Crusade:...,Kurt Vonnegut,0440180295,145,0.989927
8,1984,4,The Catcher in the Rye,J.D. Salinger,0316769487,340,0.989317
9,1984,5,Lord of the Flies,William Gerald Golding,0399501487,197,0.989191


## 3. Query sensitivity, popularity, and co-reader evidence

Low cross-query Jaccard means the ensemble responds to the input. Low popularity overlap means it is not simply reproducing the five globally most interacted items. Shared-reader counts provide direct collaborative evidence; very large lift based on only one shared reader is fragile.

In [23]:
recommendation_sets = {
    query_isbn: set(group[ITEM_COLUMN])
    for query_isbn, group in all_recommendations.groupby("query ISBN")
}

popularity_rows = []
for query_isbn, recommended in recommendation_sets.items():
    query_title = all_recommendations.loc[
        all_recommendations["query ISBN"] == query_isbn, "query title"
    ].iloc[0]
    popular_for_query = set(
        training_counts.drop(index=query_isbn, errors="ignore")
        .loc[lambda values: values.index.isin(metadata_isbns)]
        .nlargest(TOP_K)
        .index
    )
    popularity_rows.append({
        "query title": query_title,
        "popularity overlap@5": len(recommended & popular_for_query) / TOP_K,
        "median recommendation popularity rank": float(
            popularity_rank.reindex(list(recommended)).median()
        ),
    })
popularity_diagnostics = pd.DataFrame(popularity_rows)

jaccard_rows = []
for first_isbn, second_isbn in combinations(recommendation_sets, 2):
    first = recommendation_sets[first_isbn]
    second = recommendation_sets[second_isbn]
    union = first | second
    jaccard_rows.append({
        "first query ISBN": first_isbn,
        "second query ISBN": second_isbn,
        "top-5 Jaccard": len(first & second) / len(union) if union else 1.0,
    })
jaccard_diagnostics = pd.DataFrame(jaccard_rows)

needed_isbns = set(all_recommendations[ITEM_COLUMN]) | set(
    all_recommendations["query ISBN"]
)
reader_sets = (
    train_raw[train_raw[ITEM_COLUMN].isin(needed_isbns)]
    .drop_duplicates([USER_COLUMN, ITEM_COLUMN])
    .groupby(ITEM_COLUMN)[USER_COLUMN]
    .agg(set)
    .to_dict()
)
num_train_users = train_raw[USER_COLUMN].nunique()
association_rows = []
for _, row in all_recommendations.iterrows():
    query_readers = reader_sets.get(row["query ISBN"], set())
    candidate_readers = reader_sets.get(row[ITEM_COLUMN], set())
    shared = len(query_readers & candidate_readers)
    denominator = len(query_readers) * len(candidate_readers)
    association_rows.append({
        "query title": row["query title"],
        "recommendation": row["Book-Title"],
        "shared readers": shared,
        "co-reader lift": (
            shared * num_train_users / denominator if denominator else np.nan
        ),
    })
association_diagnostics = pd.DataFrame(association_rows)

display(popularity_diagnostics)
display(jaccard_diagnostics)
display(association_diagnostics.style.format({"co-reader lift": "{:.2f}"}))

,query title,popularity overlap@5,median recommendation popularity rank
0,The Hobbit : The Enchanting Prelude to The Lor...,0.0,104.0
1,1984,0.0,267.0
2,"I, Robot",0.0,391.0
3,Harry Potter and the Sorcerer's Stone (Harry P...,0.0,191.0
4,The Lord of the Rings (Movie Art Cover),0.0,267.0
5,ROMEO AND JULIET,0.0,203.0


,first query ISBN,second query ISBN,top-5 Jaccard
0,0345339681,0451524934,0.000000
1,0345339681,0553294385,0.000000
2,0345339681,059035342X,0.111111
3,0345339681,0618129022,0.000000
4,0345339681,0671722859,0.000000
5,0451524934,0553294385,0.000000
6,0451524934,059035342X,0.000000
7,0451524934,0618129022,0.000000
8,0451524934,0671722859,0.428571
9,0553294385,059035342X,0.111111


,query title,recommendation,shared readers,co-reader lift
0,The Lord of the Rings (Movie Art Cover),The Mists of Avalon,2,25.04
1,The Lord of the Rings (Movie Art Cover),Harry Potter and the Order of the Phoenix (Book 5),9,58.77
2,The Lord of the Rings (Movie Art Cover),Harry Potter and the Sorcerer's Stone (Book 1),3,38.08
3,The Lord of the Rings (Movie Art Cover),Harry Potter and the Chamber of Secrets (Book 2),4,51.50
4,The Lord of the Rings (Movie Art Cover),Harry Potter and the Goblet of Fire (Book 4),2,23.12
5,1984,Brave New World,19,121.51
6,1984,Animal Farm,18,88.08
7,1984,Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death,16,71.27
8,1984,The Catcher in the Rye,21,39.89
9,1984,Lord of the Flies,24,78.69


## 4. Evidence summary

Use this table as a compact model-level summary, then inspect the individual titles above. Query sensitivity alone is not sufficient: a useful singleton model should also avoid popularity collapse and have more than one or two shared readers supporting a typical pair. Sparse ISBN editions should be treated cautiously until editions are canonicalized to work IDs.

In [24]:
summary = pd.Series({
    "mean popularity overlap@5": popularity_diagnostics[
        "popularity overlap@5"
    ].mean(),
    "mean cross-query top-5 Jaccard": (
        jaccard_diagnostics["top-5 Jaccard"].mean()
        if not jaccard_diagnostics.empty
        else 1.0
    ),
    "median shared readers per recommendation": association_diagnostics[
        "shared readers"
    ].median(),
    "queries inspected": len(resolved_queries),
}, name="observed")
display(summary.to_frame())

,observed
mean popularity overlap@5,0.000000
mean cross-query top-5 Jaccard,0.065608
median shared readers per recommendation,12.500000
queries inspected,6.000000
